<a href="https://colab.research.google.com/github/Shadowspot/comfyui/blob/main/comfyuiV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
import os

# Google Drive base path (change if needed)
DRIVE_BASE = "/content/drive/MyDrive/comfyui"
COMFY_ROOT = "/root/comfy/ComfyUI"

drive.mount('/content/drive')

# Drive subfolder <-> ComfyUI folder mapping
custom_paths = {
    "checkpoints": "models/checkpoints",
    "loras": "models/loras",
    "vae": "models/vae",
    "controlnet": "models/controlnet",
    "embeddings": "models/embeddings",
    "upscale_models": "models/upscale_models",
    "clip": "models/clip",
    "clip_vision": "models/clip_vision",
    "unet": "models/unet",
    "workflows": "workflows",
    "output": "output",
}

def setup_custom_links(drive_root, path_dict, comfy_root=COMFY_ROOT):
    if not os.path.isdir(comfy_root):
        raise FileNotFoundError(
            f"{comfy_root} not found. Run the comfy install cell first."
        )

    for folder_type, sub_path in path_dict.items():
        drive_target = os.path.join(drive_root, sub_path)

        if folder_type == "workflows":
            local_source = os.path.join(comfy_root, "user", "default", "workflows")
        elif folder_type == "output":
            local_source = os.path.join(comfy_root, "output")
        else:
            local_source = os.path.join(comfy_root, "models", folder_type)

        os.makedirs(drive_target, exist_ok=True)
        os.makedirs(os.path.dirname(local_source), exist_ok=True)

        if os.path.islink(local_source) or os.path.isfile(local_source):
            os.unlink(local_source)
        elif os.path.isdir(local_source):
            os.system(f'rm -rf "{local_source}"')

        os.symlink(drive_target, local_source)
        print(f"[OK] {folder_type}: {local_source} -> {drive_target}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip install comfy-cli

In [14]:
!yes | comfy install --nvidia

# ??????????? Google Drive
setup_custom_links(DRIVE_BASE, custom_paths)



ComfyUI is already installed at the specified path: /root/comfy/ComfyUI

If you want to restore dependencies, add the '--restore' option.
[OK] checkpoints: /root/comfy/ComfyUI/models/checkpoints -> /content/drive/MyDrive/comfyui/models/checkpoints
[OK] loras: /root/comfy/ComfyUI/models/loras -> /content/drive/MyDrive/comfyui/models/loras
[OK] vae: /root/comfy/ComfyUI/models/vae -> /content/drive/MyDrive/comfyui/models/vae
[OK] controlnet: /root/comfy/ComfyUI/models/controlnet -> /content/drive/MyDrive/comfyui/models/controlnet
[OK] embeddings: /root/comfy/ComfyUI/models/embeddings -> /content/drive/MyDrive/comfyui/models/embeddings
[OK] upscale_models: /root/comfy/ComfyUI/models/upscale_models -> /content/drive/MyDrive/comfyui/models/upscale_models
[OK] clip: /root/comfy/ComfyUI/models/clip -> /content/drive/MyDrive/comfyui/models/clip
[OK] clip_vision: /root/comfy/ComfyUI/models/clip_vision -> /content/drive/MyDrive/comfyui/models/clip_vision
[OK] unet: /root/comfy/ComfyUI/models/unet

In [15]:
#!wget --content-disposition -O "/root/comfy/ComfyUI/models/checkpoints/noobaiXLNAIXL_vPred10Version.safetensors" "https://huggingface.co/misri/noobaiXLNAIXL_vPred10Version/resolve/main/noobaiXLNAIXL_vPred10Version.safetensors"

In [16]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 121856 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.2.0) over (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [17]:
!mkdir -p /root/comfy/ComfyUI/user/default/workflows
!curl -L -o /root/comfy/ComfyUI/user/default/workflows/示例.json https://raw.githubusercontent.com/4evergr8/ComfyColab/refs/heads/main/%E7%A4%BA%E4%BE%8B.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  5438  100  5438    0     0  15642      0 --:--:-- --:--:-- --:--:-- 15626


In [18]:
# Start ComfyUI in background (more stable than foreground execution)
!pkill -f 'python3 /root/comfy/ComfyUI/main.py' || true
!nohup python3 /root/comfy/ComfyUI/main.py --listen 127.0.0.1 --port 8188 --dont-print-server > /content/comfy.log 2>&1 &

import socket, time
for i in range(120):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    ok = s.connect_ex(("127.0.0.1", 8188)) == 0
    s.close()
    if ok:
        print("[OK] ComfyUI is running on 127.0.0.1:8188")
        break
    time.sleep(1)
else:
    raise RuntimeError("ComfyUI did not start in time. Check /content/comfy.log")


^C
[OK] ComfyUI is running on 127.0.0.1:8188


In [19]:
# Verify key links (should point to /content/drive/...)
!ls -la /root/comfy/ComfyUI/models/checkpoints | head
!ls -la /root/comfy/ComfyUI/models/loras | head
!ls -la /root/comfy/ComfyUI/output | head


lrwxrwxrwx 1 root root 49 Mar  3 06:59 /root/comfy/ComfyUI/models/checkpoints -> /content/drive/MyDrive/comfyui/models/checkpoints
lrwxrwxrwx 1 root root 43 Mar  3 06:59 /root/comfy/ComfyUI/models/loras -> /content/drive/MyDrive/comfyui/models/loras
lrwxrwxrwx 1 root root 37 Mar  3 06:59 /root/comfy/ComfyUI/output -> /content/drive/MyDrive/comfyui/output


In [20]:
# Start Cloudflare tunnel and print public URL
!pkill -f 'cloudflared tunnel --url http://127.0.0.1:8188' || true
!nohup cloudflared tunnel --url http://127.0.0.1:8188 --protocol http2 --no-autoupdate > /content/cloudflared.log 2>&1 &

import re, time
from pathlib import Path

url = None
log = Path('/content/cloudflared.log')
for _ in range(90):
    if log.exists():
        txt = log.read_text(errors='ignore')
        m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', txt)
        if m:
            url = m.group(0)
            break
    time.sleep(1)

if url:
    print(f"[OK] Open ComfyUI: {url}")
else:
    print("[WARN] Tunnel URL not found yet. Check: /content/cloudflared.log")


^C
[OK] Open ComfyUI: https://elizabeth-phantom-vampire-ordinance.trycloudflare.com
